# Lab 09 — Sintonia de PID: Ziegler–Nichols, CHR e método lambda (IMC)

**Unidade IV — Projeto, sintonia e implementação de PID** · conteúdo 4.2 do PPC

**Objetivos:**
1. Aplicar Ziegler–Nichols de malha fechada usando $(K_u, T_u)$ obtidos nos Labs 05/07;
2. Aplicar ZN de malha aberta e CHR a partir da curva de reação (modelo FOPDT do Lab 02);
3. Aplicar o método lambda (IMC) e explorar o compromisso robustez × velocidade;
4. Comparar as sintonias em seguimento de referência **e** rejeição de perturbação.

**Referências:** Åström & Hägglund, *Advanced PID Control*, caps. 6–7 · Åström & Murray (FBS), cap. 11 · Ogata, cap. 10.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    import control as ct
    print("python-control", ct.__version__)
except ImportError:
    %pip install control
    import control as ct

def pid_tf(Kp, Ti=np.inf, Td=0.0, N=10):
    """PID ISA com derivada filtrada (mesma função do Lab 08)."""
    C = ct.tf([Kp], [1])
    if np.isfinite(Ti):
        C = C + ct.tf([Kp], [Ti, 0])
    if Td > 0:
        C = C + ct.tf([Kp * Td, 0], [Td / N, 1])
    return C

## 1. Planta A (3ª ordem) — sintonia por Ziegler–Nichols de malha fechada

Do Lab 05/07: $K_u = 90$, $T_u = 1{,}68$ s.

| Controlador | $K_p$ | $T_i$ | $T_d$ |
|---|---|---|---|
| P | $0{,}5 K_u$ | — | — |
| PI | $0{,}45 K_u$ | $T_u/1{,}2$ | — |
| PID | $0{,}6 K_u$ | $T_u/2$ | $T_u/8$ |

In [ ]:
G_A = ct.tf([1], np.polymul(np.polymul([1, 1], [1, 2]), [1, 4]))
Ku, Tu = 90.0, 1.68

sintonias_zn = {
    'P   (ZN)': pid_tf(0.5 * Ku),
    'PI  (ZN)': pid_tf(0.45 * Ku, Tu / 1.2),
    'PID (ZN)': pid_tf(0.6 * Ku, Tu / 2, Tu / 8),
}

t = np.linspace(0, 10, 1000)
plt.figure(figsize=(9, 5))
for nome, C in sintonias_zn.items():
    T_cl = ct.feedback(C * G_A, 1)
    resp = ct.step_response(T_cl, t)
    info = ct.step_info(T_cl)
    plt.plot(resp.time, resp.outputs, lw=2,
             label=f"{nome}: Mp = {info['Overshoot']:.0f}%, ts = {info['SettlingTime']:.1f}s")
plt.axhline(1, color='gray', ls='--')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Ziegler–Nichols de malha fechada (planta A)')
plt.legend(); plt.grid(True)
plt.show()

**Diagnóstico esperado:** ZN entrega respostas rápidas porém **agressivas** (sobressinal de
40–60 % é típico — o método foi projetado para razão de decaimento 1/4, priorizando rejeição
de perturbação). Auditoria de margens:

In [ ]:
for nome, C in sintonias_zn.items():
    gm, pm, _, _ = ct.margin(C * G_A)
    print(f"{nome}: PM = {pm:5.1f} graus | GM = {20*np.log10(gm):5.2f} dB")

PM ≈ 25–35° explica o sobressinal (regra $\zeta \approx$ PM/100). ZN é **ponto de partida**:
na prática, reduz-se $K_p$ (ou aumenta-se $T_i$) até PM ≥ 45°.

## 2. Planta B (FOPDT) — curva de reação: ZN de malha aberta e CHR

Usamos a planta identificada no Lab 02: $G_B(s) = \dfrac{3\,e^{-1{,}5s}}{4s + 1}$
($K = 3$, $\tau = 4$, $\theta = 1{,}5$).

| Método (PI) | $K_p$ | $T_i$ |
|---|---|---|
| ZN aberto | $0{,}9\,\tau/(K\theta)$ | $3{,}33\,\theta$ |
| CHR 0% sobressinal (servo) | $0{,}35\,\tau/(K\theta)$ | $1{,}17\,\tau$ |

| Método (PID) | $K_p$ | $T_i$ | $T_d$ |
|---|---|---|---|
| ZN aberto | $1{,}2\,\tau/(K\theta)$ | $2\theta$ | $0{,}5\theta$ |
| CHR 0% (servo) | $0{,}6\,\tau/(K\theta)$ | $\tau$ | $0{,}5\theta$ |

In [ ]:
K_B, tau_B, theta_B = 3.0, 4.0, 1.5
num_p, den_p = ct.pade(theta_B, 5)
G_B = ct.tf([K_B], [tau_B, 1]) * ct.tf(num_p, den_p)

a = K_B * theta_B / tau_B   # parâmetro auxiliar das tabelas

sintonias_B = {
    'PI  ZN-aberto': pid_tf(0.9 / a, 3.33 * theta_B),
    'PI  CHR-0%':    pid_tf(0.35 / a, 1.17 * tau_B),
    'PID ZN-aberto': pid_tf(1.2 / a, 2 * theta_B, 0.5 * theta_B),
    'PID CHR-0%':    pid_tf(0.6 / a, tau_B, 0.5 * theta_B),
}

t_B = np.linspace(0, 30, 3000)
plt.figure(figsize=(9, 5))
for nome, C in sintonias_B.items():
    T_cl = ct.feedback(C * G_B, 1)
    resp = ct.step_response(T_cl, t_B)
    plt.plot(resp.time, resp.outputs, lw=2, label=nome)
plt.axhline(1, color='gray', ls='--')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Curva de reação: ZN agressivo × CHR conservador (planta B)')
plt.legend(); plt.grid(True)
plt.show()

CHR-0 % cumpre o prometido: sem sobressinal, ao custo de resposta mais lenta.
A escolha entre métodos é uma decisão de **projeto**, não de gosto: processos que não toleram
ultrapassagem (temperatura de reator, pH) pedem CHR/lambda; malhas de rejeição rápida toleram ZN.

### 2.1 O método gráfico da tangente (como se faz na bancada)

Na prática, $\tau$ e $\theta$ não vêm de fórmulas: vêm da **curva de reação** medida. O método
clássico de ZN (FBS, seç. 11.3) traça a tangente no ponto de inclinação máxima do degrau:
a interseção com o eixo do tempo dá $\theta$ (e o parâmetro $a$ = interseção com $t = 0$,
em módulo). Automatizamos a construção:

In [ ]:
resp_B = ct.step_response(G_B, np.linspace(0, 25, 2500))
slope = np.diff(resp_B.outputs) / np.diff(resp_B.time)
i_mx = np.argmax(slope)                       # ponto de inclinacao maxima
t_mx, y_mx, s_mx = resp_B.time[i_mx], resp_B.outputs[i_mx], slope[i_mx]

theta_est = t_mx - y_mx / s_mx                # tangente cruza y=0 em t = theta
a_zn = s_mx * theta_est                       # parametro 'a' das tabelas ZN
K_est = resp_B.outputs[-1]                    # ganho estatico
tau_est = K_est / s_mx                        # aproximacao: inclinacao max ~ K/tau

plt.figure(figsize=(9, 5))
plt.plot(resp_B.time, resp_B.outputs, lw=2, label='curva de reação')
t_tan = np.linspace(0, 2.5 * t_mx, 50)
plt.plot(t_tan, y_mx + s_mx * (t_tan - t_mx), 'r-', lw=1.5, label='tangente na máx. inclinação')
plt.plot(t_mx, y_mx, 'ro')
plt.axhline(0, color='k', lw=0.5); plt.axhline(K_est, color='gray', ls=':')
plt.ylim(-0.5, 3.3)
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title(f'Método da tangente: θ ≈ {theta_est:.2f} s, τ ≈ {tau_est:.2f} s '
          f'(gabarito: θ = {theta_B}, τ = {tau_B})')
plt.legend(); plt.grid(True)
plt.show()

**No projeto final este será o procedimento real:** degrau na bancada → tangente (ou método
dos dois pontos do Lab 02, mais robusto a ruído) → $(K, \tau, \theta)$ → tabelas.

## 3. Método lambda (IMC): um botão com significado físico

Para FOPDT com PI: $K_p = \dfrac{\tau}{K(\lambda + \theta)}$, $T_i = \tau$.
$\lambda$ é a constante de tempo **desejada** da malha fechada.

In [ ]:
plt.figure(figsize=(9, 5))
for lam_fator, cor in [(0.5, 'C3'), (1.0, 'C1'), (2.0, 'C2')]:
    lam = lam_fator * tau_B
    C = pid_tf(tau_B / (K_B * (lam + theta_B)), tau_B)
    T_cl = ct.feedback(C * G_B, 1)
    resp = ct.step_response(T_cl, t_B)
    gm, pm, _, _ = ct.margin(C * G_B)
    plt.plot(resp.time, resp.outputs, cor, lw=2,
             label=f'λ = {lam_fator}τ  (PM = {pm:.0f}°)')
plt.axhline(1, color='gray', ls='--')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Método lambda: velocidade × robustez em um único parâmetro')
plt.legend(); plt.grid(True)
plt.show()

## 4. O teste que separa os métodos: rejeição de perturbação

Comparar sintonias **apenas** ao degrau de referência é enganoso. Simulamos o cenário completo:
degrau de referência em $t = 0$ e **perturbação na entrada da planta** em $t = 15$ s.

In [ ]:
candidatos = {
    'PID ZN-aberto': sintonias_B['PID ZN-aberto'],
    'PID CHR-0%': sintonias_B['PID CHR-0%'],
    'PI  lambda (λ=τ/2)': pid_tf(tau_B / (K_B * (0.5 * tau_B + theta_B)), tau_B),
}

t_c = np.linspace(0, 30, 3000)
plt.figure(figsize=(10, 5))
for nome, C in candidatos.items():
    T_ry = ct.feedback(C * G_B, 1)          # referência -> saída
    T_dy = ct.feedback(G_B, C)              # perturbação de entrada -> saída
    y_r = ct.step_response(T_ry, t_c).outputs
    # perturbação de -0.5 na entrada da planta a partir de t = 15 s
    d = -0.5 * (t_c >= 15)
    y_d = ct.forced_response(T_dy, t_c, d).outputs
    plt.plot(t_c, y_r + y_d, lw=2, label=nome)
plt.axhline(1, color='gray', ls='--')
plt.axvline(15, color='gray', ls=':', label='perturbação em t = 15 s')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Servo (t = 0) + regulatório (t = 15 s): o quadro completo')
plt.legend(); plt.grid(True)
plt.show()

**Observe o conflito:** o ZN (agressivo) rejeita a perturbação mais rápido; o CHR (suave) tem o
melhor degrau de referência. Nenhuma sintonia 1DOF vence nos dois — **a motivação exata do
controle 2DOF do Lab 11.**

## 5. Além das tabelas: sintonia por *loop shaping*

As tabelas dão o ponto de partida; o refinamento profissional é feito **moldando a FT de
malha** $L(s) = C(s)G(s)$ no Bode (metodologia central do CDS 110, FBS cap. 12). As
restrições típicas sobre $|L(j\omega)|$:

- **ganho alto em baixa frequência** → bom seguimento e rejeição de perturbação
  (o "I" garante $|L| \to \infty$ quando $\omega \to 0$);
- **ganho baixo em alta frequência** → não amplificar ruído (o filtro derivativo ajuda);
- **cruzamento de ganho** ($|L| = 1$) na banda desejada → define a velocidade;
- **inclinação suave (~ −20 dB/dec) no cruzamento** → como ganho e fase não são
  independentes (relação de Bode), inclinação suave ⟹ fase alta ⟹ **PM grande**.

Interpretação das ações no Bode de $C$: o zero $1/T_i$ delimita onde o "I" atua (esquerda),
o zero $1/T_d$ onde o "D" adianta fase (direita), e $K_p$ desliza a curva toda verticalmente.
Vamos moldar $L$ para a planta A com alvo de cruzamento em $\omega_{gc} \approx 2$ rad/s e
PM ≥ 50°:

In [ ]:
# projeto por loop shaping (iterativo, mas mostrado ja no resultado):
# 1) Td posicionado ~ na frequencia de cruzamento desejada para adiantar fase la;
# 2) Ti ~ 5x acima de Td para o "I" nao roubar fase no cruzamento;
# 3) Kp ajustado para |L(j w_gc)| = 1.
Td_ls = 0.5                 # 1/Td = 2 rad/s = regiao do cruzamento
Ti_ls = 5 * Td_ls           # 1/Ti = 0.4 rad/s, bem abaixo do cruzamento
C_ls_semKp = pid_tf(1.0, Ti_ls, Td_ls, N=10)
w_gc_alvo = 2.0
Kp_ls = 1.0 / np.abs((C_ls_semKp * G_A)(1j * w_gc_alvo))
C_ls = pid_tf(Kp_ls, Ti_ls, Td_ls, N=10)
L_ls = C_ls * G_A

gm, pm, wpc, wgc = ct.margin(L_ls)
print(f"Loop shaping: Kp = {Kp_ls:.1f}, Ti = {Ti_ls}, Td = {Td_ls}")
print(f"Cruzamento em {wgc:.2f} rad/s | PM = {pm:.1f} graus | GM = {20*np.log10(gm):.1f} dB")

# Bode de L com margens + comparacao com o PID-ZN da secao 1
L_zn = sintonias_zn['PID (ZN)'] * G_A
ct.bode_plot([L_ls, L_zn], np.logspace(-2, 2, 500),
             label=['L loop-shaping', 'L Ziegler-Nichols'], margins=True)
plt.show()

In [ ]:
# validacao no tempo: servo + regulatorio
t_ls = np.linspace(0, 10, 1000)
plt.figure(figsize=(9, 5))
for C_k, nome in [(C_ls, 'loop shaping'), (sintonias_zn['PID (ZN)'], 'ZN')]:
    T_ry_k = ct.feedback(C_k * G_A, 1)
    T_dy_k = ct.feedback(G_A, C_k)
    y_k = ct.step_response(T_ry_k, t_ls).outputs \
        + ct.forced_response(T_dy_k, t_ls, -20.0 * (t_ls >= 6)).outputs
    info = ct.step_info(T_ry_k)
    plt.plot(t_ls, y_k, lw=2, label=f"{nome} (Mp = {info['Overshoot']:.0f}%)")
plt.axhline(1, color='gray', ls='--'); plt.axvline(6, color='gray', ls=':')
plt.xlabel('Tempo [s]'); plt.ylabel('y(t)')
plt.title('Loop shaping × ZN: mesma banda, muito mais margem (e menos sobressinal)')
plt.legend(); plt.grid(True)
plt.show()

O controlador moldado atinge sobressinal e margens muito melhores que o ZN **na mesma
velocidade de malha** — porque projetamos diretamente a grandeza que importa ($L$ no
cruzamento) em vez de usar uma receita cega.

## 6. Procedimento de sintonia recomendado no curso

1. Identificar a planta (degrau → FOPDT, Lab 02) **ou** obter $(K_u, T_u)$ pelo relé (Lab 07);
2. Sintonia inicial por tabela (lambda com $\lambda = \tau$ como padrão conservador);
3. Auditar: PM ≥ 45°, GM ≥ 6 dB, $M_s \le 2$;
4. Simular servo + regulatório + saturação do atuador;
5. Refinar por **loop shaping** ($T_d$ no cruzamento, $T_i \approx 5T_d$, $K_p$ pela banda)
   conforme a prioridade do processo;
6. Só então ir para a bancada.

---
> **🖼️ Figuras de apoio nos livros:**
> - Ogata, **Figura 8.2** — ensaio de resposta ao degrau unitário da planta (1º método de Ziegler–Nichols). Cap. 8, §8.2, **p. 523** (p. 534 do PDF).
> - Ogata, **Figura 8.3** — curva de resposta em forma de S, com a tangente no ponto de inflexão definindo o atraso L e a constante T. Cap. 8, §8.2, **p. 523** (p. 534 do PDF).
> - Ogata, **Tabela 8.1** — regra de sintonia de Ziegler–Nichols pelo 1º método (curva de reação: L e T). Cap. 8, §8.2, **p. 524** (p. 535 do PDF).
> - Ogata, **Tabela 8.2** — regra de sintonia de Ziegler–Nichols pelo 2º método ($K_{cr}$ e $P_{cr}$). Cap. 8, §8.2, **p. 525** (p. 536 do PDF).
> - Penedo, **Figura 9.4** — estratégia de controle do método de Ziegler–Nichols do período crítico (e o método da resposta ao degrau nas figuras do §9.2, pp. 91–92 do PDF). Cap. 9, p. 93 do arquivo PDF.
> - Transparências CDS 110 **L9-1**, **slide 13** — 'PID Tuning': os dois métodos de ZN com as duas tabelas.
> - Transparências CDS 110 **L9-1**, **slides 2–4** — restrições de loop shaping sobre $|L(j\omega)|$.

## Exercícios (relatório do Lab 09)

**E1.** Complete a auditoria: monte uma tabela com PM, GM, $M_s$, $M_p$, $t_s$ e IAE
($\int|e|dt$, calcule numericamente com `np.trapezoid`) para todas as sintonias da planta B.
Qual venceria para um processo térmico que não tolera sobressinal?

**E2.** Para a planta A, parta do PID-ZN e ajuste **um parâmetro por vez** até PM ≥ 45°
mantendo $t_s \le 3$ s. Registre cada passo.

**E3.** Sintonize por lambda a planta FOPDT que você identificou no E4 do Lab 07 e valide
em servo + regulatório.

**E4.** Investigue a sensibilidade da sintonia a erros de modelo: aplique o PID CHR da planta B
a uma planta com $\tau$ 30 % menor e $\theta$ 30 % maior. A malha continua estável? E o ZN?

**E5.** Refaça o loop shaping da Seção 5 com alvo $\omega_{gc} = 3$ rad/s. Até onde é possível
acelerar mantendo PM ≥ 45°? Relacione o limite encontrado com a fase da planta em alta
frequência (3 polos ⟹ fase → −270°).

In [ ]:
# E1 — sua solução aqui

In [ ]:
# E2 — sua solução aqui

In [ ]:
# E3 — sua solução aqui

In [ ]:
# E4 — sua solução aqui

In [ ]:
# E5 — sua solução aqui